# Characterizing Multi-User XR Traffic from Passive Network Measurements
## End-to-End Analysis Pipeline

This notebook implements the complete empirical analysis pipeline for the **Characterizing Multi-User XR Traffic from Passive Network Measurements** paper. It processes passive packet captures collected at the edge gateway VM during an operational 9-user Meta Quest 3 session running **MetaClassroom** over Wi-Fi 6.

### Pipeline Stages:
1. **Dataset Ingestion & Classification**: Zero-dependency extraction of transport metadata and UDP application payloads from `./data/`.
2. **Traffic Volume & Directionality Audit**: Directional byte and packet breakdown (verifying the ~80% downlink volume asymmetry).
3. **Protocol Inference & RTT Decomposition**: Sub-second keep-alive identification, loss audit, and directional latency decomposition (WAN floor vs. server processing cohorts).
4. **Engine Tick Modeling (4-Component GMM)**: Multi-harmonic inter-arrival modeling isolating 65 Hz, 33 Hz, and 22 Hz cadences.
5. **Payload Modeling (Log-Normal)**: Parametric characterization of scene state updates vs. compact 6-DoF telemetry.
6. **Goodness-of-Fit Validation**: Two-sample Kolmogorov-Smirnov (K-S) tests.
7. **Session Timeline & Scenario Analysis**: Phase-by-phase throughput stability across the 43.4-minute capture.
8. **Figure Compilation**: Rendering publication-grade figures (`figures/heartbeat_stats.png`, `figures/fig_throughput_minmax_accessible.png`, `figures/bidirectional_3panel.png`).

In [1]:
# %% Step 0: Imports, Environment Setup, and Helpers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from helpers import (
    DATA_DIR,
    FIGURES_DIR,
    HEADSET_IPS,
    HEADSET_LABELS,
    SESSION_PHASES,
    extract_game_traffic,
    extract_heartbeats_and_rtt,
    compute_throughput_timeseries,
    fit_downstream_gmm,
    fit_payload_distributions,
    evaluate_ks_tests,
    set_publication_style,
)
from compile_figures import (
    plot_figure_2_heartbeat_stats,
    plot_figure_3_throughput,
    plot_figure_4_bidirectional_models,
)

set_publication_style()
print("Helpers and plotting modules loaded successfully.")

Helpers and plotting modules loaded successfully.


### Step 1: Dataset Ingestion & Traffic Extraction

The trace spans **43.4 minutes** and contains **1,003,448 UDP packets** on port 30780 exchanged between the local wireless subnet (`192.168.10.0/24`) and the cloud server (`172.234.115.99`) in Stockholm, Sweden.

`extract_game_traffic()` parses the PCAP stream directly without external dependencies, classifies packets by direction (`upstream` vs. `downstream`), and isolates keep-alive frames from continuous state updates.

In [2]:
# %% Step 1: Load and Classify Game Traffic
df_all = extract_game_traffic()

t0 = df_all["timestamp"].min()
t_end = df_all["timestamp"].max()
duration_min = (t_end - t0) / 60.0

print(f"Dataset Summary:")
print(f"  Total UDP Game Packets : {len(df_all):,}")
print(f"  Session Duration       : {duration_min:.2f} minutes")
print(f"  Active Headsets (N=9)  : {[ip.split('.')[-1] for ip in sorted(df_all['headset_ip'].unique())]}")
df_all.head(6)

Loading cached game traffic ...
Dataset Summary:
  Total UDP Game Packets : 1,003,448
  Session Duration       : 43.41 minutes
  Active Headsets (N=9)  : ['21', '22', '23', '24', '26', '27', '28', '29', '30']


,timestamp,headset_ip,direction,payload_len,is_heartbeat,msg_type
0,1.780919e+09,192.168.10.27,upstream,36,False,5
1,1.780919e+09,192.168.10.27,downstream,15,False,6
2,1.780919e+09,192.168.10.27,upstream,18,False,1
3,1.780919e+09,192.168.10.27,upstream,18,False,1
4,1.780919e+09,192.168.10.27,downstream,13,False,2
5,1.780919e+09,192.168.10.27,downstream,93,False,12


### Step 2: Traffic Volume & Directionality Audit

MetaClassrooms traffic displays strong directional asymmetry. While packet counts are balanced (~500k upstream vs. ~500k downstream), **downstream traffic accounts for ~80% of total transferred byte volume**.

- **Upstream**: Clients upload lightweight 6-DoF tracking telemetry and audio (median 51 B).
- **Downstream**: The cloud server replicates aggregated world states for all participants (median 225-228 B).

In [3]:
# %% Step 2: Audit Volume, Byte Counts, and Asymmetries
vol_summary = df_all.groupby(["direction", "is_heartbeat"]).agg(
    packet_count=("payload_len", "count"),
    total_bytes=("payload_len", "sum"),
    mean_bytes=("payload_len", "mean"),
    median_bytes=("payload_len", "median")
).reset_index()

down_bytes = df_all[df_all["direction"] == "downstream"]["payload_len"].sum()
up_bytes = df_all[df_all["direction"] == "upstream"]["payload_len"].sum()
total_bytes = down_bytes + up_bytes

print("--- Traffic Breakdown by Direction and Frame Type ---")
print(vol_summary.to_string(index=False))
print(f"\nDownstream Volume Share: {down_bytes / total_bytes * 100.0:.2f}%")
print(f"Upstream Volume Share  : {up_bytes / total_bytes * 100.0:.2f}%")

--- Traffic Breakdown by Direction and Frame Type ---
 direction  is_heartbeat  packet_count  total_bytes  mean_bytes  median_bytes
downstream         False        467935    129329682  276.383861         232.0
downstream          True         32584       226896    6.963418           3.0
  upstream         False        470720     31806676   67.570267          51.0
  upstream          True         32209       579762   18.000000          18.0

Downstream Volume Share: 80.00%
Upstream Volume Share  : 20.00%


### Step 3: Passive Protocol Inference & Directional RTT Decomposition

We identify sub-second control pairs using opcode matching: `0x03` (Ping) and `0x04` (Pong).

- **Client-initiated pings ($RTT_{c2s}$)**: Headset transmits an 18 B ping (`0x03`). The server replies with a 3 B or 11 B pong (`0x04`). Measures WAN propagation + server-side OS scheduling and game loop processing.
- **Server-initiated pings ($RTT_{s2c}$)**: Server dispatches a 3 B or 11 B ping (`0x03`). The headset replies with an 18 B pong (`0x04`). Isolates the local Wi-Fi 6 wireless hop and Quest 3 network turnaround.

Matching transaction IDs across the gateway yields **31,850 matched RTT samples** (15,943 client pings and 15,907 server pings).

In [4]:
# %% Step 3: Match Transactions, Extract RTTs, and Audit Packet Loss
df_rtt, df_loss = extract_heartbeats_and_rtt()

c2s_rtt = df_rtt[df_rtt["direction"] == "c2s"]
s2c_rtt = df_rtt[df_rtt["direction"] == "s2c"]

print(f"Matched RTT Samples : {len(df_rtt):,}")
print(f"  Client Pings (c2s): {len(c2s_rtt):,}")
print(f"  Server Pings (s2c): {len(s2c_rtt):,}")

# Latency floor & cohort decomposition
cohort1_ips = ["192.168.10.21", "192.168.10.22", "192.168.10.26", "192.168.10.28"]
cohort2_ips = ["192.168.10.23", "192.168.10.24", "192.168.10.27", "192.168.10.29", "192.168.10.30"]

print(f"\nLatency Decomposition:")
print(f"  Server Ping Floor (s2c) : {s2c_rtt['rtt_ms'].median():.2f} ms  (WAN floor ~3.8 ms)")
print(f"  Client Ping Cohort 1     : {c2s_rtt[c2s_rtt['headset_ip'].isin(cohort1_ips)]['rtt_ms'].median():.2f} ms  (~7 ms cohort)")
print(f"  Client Ping Cohort 2     : {c2s_rtt[c2s_rtt['headset_ip'].isin(cohort2_ips)]['rtt_ms'].median():.2f} ms  (~12 ms cohort)")

print("\n--- Packet Loss Audit (Unanswered Probes) ---")
print(df_loss.sort_values(by=["headset_ip", "direction"]).to_string(index=False))

Matched RTT Samples : 31,850
  Client Pings (c2s): 16,143
  Server Pings (s2c): 15,707

Latency Decomposition:
  Server Ping Floor (s2c) : 3.97 ms  (WAN floor ~3.8 ms)
  Client Ping Cohort 1     : 7.06 ms  (~7 ms cohort)
  Client Ping Cohort 2     : 12.06 ms  (~12 ms cohort)

--- Packet Loss Audit (Unanswered Probes) ---
   headset_ip direction  pings_sent  pongs_received  lost_heartbeats  loss_pct
192.168.10.21       c2s        2196            2196                0     0.000
192.168.10.21       s2c        2248            2190               58     2.580
192.168.10.22       c2s        1367            1367                0     0.000
192.168.10.22       s2c        1418            1302              116     8.181
192.168.10.23       c2s        1275            1275                0     0.000
192.168.10.23       s2c        1329            1146              183    13.770
192.168.10.24       c2s        2103            2102                1     0.048
192.168.10.24       s2c        2095          

### Step 4: Decomposition of Engine Tick Cadences (4-Component GMM)

Downstream packet inter-arrivals $\tau_d$ exhibit distinct, multi-harmonic peaks corresponding to server scheduling loops:

$$f(\tau_d) = \sum_{k=1}^4 \frac{w_k}{\sqrt{2\pi\sigma_k^2}} \exp\left(-\frac{(\tau_d - \mu_k)^2}{2\sigma_k^2}\right)$$

- **Mode 1 (~65 Hz, $\mu_1 \approx 15.4$ ms)**: High-priority avatar inverse kinematics (IK) and head/hand transforms.
- **Mode 2 (~33 Hz, $\mu_2 \approx 30.5$ ms)**: Master Unity tick rate loop.
- **Mode 3 (Stochastic Tail, $\mu_3 \approx 33.9$ ms, $\sigma_3 \approx 30.8$ ms)**: Over-the-air Wi-Fi queuing and MAC-layer retries.
- **Mode 4 (~22 Hz, $\mu_4 \approx 45.6$ ms)**: Sub-harmonic cadence for lower-priority environmental scene replication.

In [5]:
# %% Step 4: Fit 4-Component GMM on Downstream Inter-Arrival Times
gmm_model, gmm_modes_df, delta_down_clean, gmm_params = fit_downstream_gmm(df_all)

print("--- Fitted 4-Component GMM Modes ---")
print(gmm_modes_df.to_string(index=False))

modes_deterministic_weight = gmm_modes_df.loc[gmm_modes_df['Mode'].isin(['Mode 1', 'Mode 2', 'Mode 4']), 'Weight (%)'].sum()
print(f"\nTotal deterministic update share (Modes 1, 2, 4): {modes_deterministic_weight:.2f}%")

--- Fitted 4-Component GMM Modes ---
  Mode  Mean (ms)  Std (ms)  Weight (%)  Approx Freq (Hz)
Mode 1      15.36      0.13       31.96              65.1
Mode 2      30.50      0.14       25.67              32.8
Mode 3      33.89     30.81        7.21              29.5
Mode 4      45.64      0.15       35.16              21.9

Total deterministic update share (Modes 1, 2, 4): 92.79%


### Step 5: Application Payload Size Modeling

Excluding keep-alives and control frames reveals application payload dynamics:
- **Upstream**: Centered on compact 51 B median (6-DoF telemetry poses).
- **Downstream**: Follows a 3-parameter Log-Normal distribution ($s=0.481$, $\text{loc}=-122.6$, $\text{scale}=356.4$) with median 232 B (P95 = 645 B) for full room state replication.

In [6]:
# %% Step 5: Model Application Payload Distributions
lognorm_params, s_down, s_up = fit_payload_distributions(df_all)
shape_d, loc_d, scale_d = lognorm_params

print("--- Application Payload Distributions ---")
print(f"Downstream (Bytes): Mean = {np.mean(s_down):.1f}, Median = {np.median(s_down):.1f}, P95 = {np.percentile(s_down, 95):.1f}, Std = {np.std(s_down):.1f}")
print(f"  Log-Normal Fit   -> shape (s)={shape_d:.3f}, loc={loc_d:.1f}, scale={scale_d:.1f}")
print(f"Upstream (Bytes)  : Mean = {np.mean(s_up):.1f}, Median = {np.median(s_up):.1f}, P95 = {np.percentile(s_up, 95):.1f}, Std = {np.std(s_up):.1f}")

--- Application Payload Distributions ---
Downstream (Bytes): Mean = 276.4, Median = 232.0, P95 = 645.0, Std = 192.6
  Log-Normal Fit   -> shape (s)=0.481, loc=-122.6, scale=356.4
Upstream (Bytes)  : Mean = 67.6, Median = 51.0, P95 = 203.0, Std = 63.7


### Step 6: Kolmogorov-Smirnov (K-S) Goodness-of-Fit Validation

We evaluate the empirical cumulative distribution functions against our analytical models:
1. **Inter-Arrival GMM**: $D = 0.0290$ (max CDF deviation 2.90%).
2. **Payload Log-Normal**: $D = 0.0330$ (max CDF deviation 3.30%).

In [7]:
# %% Step 6: Perform Two-Sample K-S Tests
(ks_gmm, p_gmm), (ks_lognorm, p_lognorm) = evaluate_ks_tests(
    delta_down_clean, gmm_params, s_down, lognorm_params
)

print("--- K-S Test Goodness-of-Fit Results ---")
print(f"Downstream Inter-Arrival GMM : K-S Statistic (D) = {ks_gmm:.4f}, p-value = {p_gmm:.4e}")
print(f"  Max CDF Deviation          : {ks_gmm * 100:.2f}%")
print(f"Downstream Payload LogNormal : K-S Statistic (D) = {ks_lognorm:.4f}, p-value = {p_lognorm:.4e}")
print(f"  Max CDF Deviation          : {ks_lognorm * 100:.2f}%")

--- K-S Test Goodness-of-Fit Results ---
Downstream Inter-Arrival GMM : K-S Statistic (D) = 0.0290, p-value = 4.2771e-04
  Max CDF Deviation          : 2.90%
Downstream Payload LogNormal : K-S Statistic (D) = 0.0330, p-value = 3.7318e-05
  Max CDF Deviation          : 3.30%


### Step 7: Session Timeline & Scenario Phase Analysis

The 43.4-minute deployment comprises four sequential activity phases:
1. **Phase 0: Room Join and Setup (0–14 min)**: Headsets sequentially connect.
2. **Scenario 1: Lecture (14–27 min)**: 3D presentation by external instructor. Participants act as stationary listeners; downstream throughput stabilizes (~80–100 kbps per headset) and variance drops.
3. **Intermission (27–29 min)**: Transition break.
4. **Scenario 2: Task Execution (29–36 min)**: Interactive exploration locating hidden objects.

In [8]:
# %% Step 7: Compute Phase Statistics and Timeline Throughput
per_hs, stats_hs, total_agg, active_hs = compute_throughput_timeseries(df_all)

print("--- Phase-by-Phase Throughput Summary ---")
for phase_name, start_m, end_m, _ in SESSION_PHASES:
    sub_up = stats_hs[(stats_hs["direction"] == "upstream") & (stats_hs["t_min"] >= start_m) & (stats_hs["t_min"] < end_m)]
    sub_down = stats_hs[(stats_hs["direction"] == "downstream") & (stats_hs["t_min"] >= start_m) & (stats_hs["t_min"] < end_m)]
    
    print(f"{phase_name:<28} ({start_m:.0f}–{end_m:.0f} min):")
    print(f"  Downstream : Mean = {sub_down['mean'].mean():.1f} kbps, Std = {sub_down['std'].mean():.1f} kbps")
    print(f"  Upstream   : Mean = {sub_up['mean'].mean():.1f} kbps, Std = {sub_up['std'].mean():.1f} kbps")

--- Phase-by-Phase Throughput Summary ---
Phase 0: Room Join           (0–14 min):
  Downstream : Mean = 33.2 kbps, Std = 19.6 kbps
  Upstream   : Mean = 12.9 kbps, Std = 10.4 kbps
Scenario 1: Lecture          (14–27 min):
  Downstream : Mean = 88.7 kbps, Std = 5.7 kbps
  Upstream   : Mean = 15.9 kbps, Std = 4.3 kbps
Intermission                 (27–29 min):
  Downstream : Mean = 72.0 kbps, Std = 10.3 kbps
  Upstream   : Mean = 19.1 kbps, Std = 11.1 kbps
Scenario 2: Task Execution   (29–36 min):
  Downstream : Mean = 54.2 kbps, Std = 18.3 kbps
  Upstream   : Mean = 19.7 kbps, Std = 8.3 kbps


### Step 8: Compiling Paper Figures

Generates the three paper figures into the `figures/` directory:
- **Figure 5**: `figures/heartbeat_stats.png` (Directional RTT bar chart & scaling load stability)
- **Figure 5**: `figures/fig_throughput_minmax_accessible.png` (3-panel throughput timeline with phase shading)
- **Figure 5**: `figures/bidirectional_3panel.png` (PDF, CDF, and payload distributions with model fits)

In [9]:
# %% Step 8: Render and Save All Figures
plot_figure_2_heartbeat_stats(df_rtt, df_all, FIGURES_DIR)
plot_figure_3_throughput(df_all, FIGURES_DIR)
plot_figure_4_bidirectional_models(df_all, FIGURES_DIR)

print("All paper figures have been compiled into figures/ directory.")

Generating Figure 2: figures/heartbeat_stats.png ...
  -> Saved "heartbeat_stats.png" and "heartbeat_stats.pdf"
Generating Figure 3: figures/fig_throughput_minmax_accessible.png ...
  -> Saved "fig_throughput_minmax_accessible.png" and "fig_throughput_minmax_accessible.pdf"
Generating Figure 4: figures/bidirectional_3panel.png ...
  -> Saved "bidirectional_3panel.png" and "bidirectional_3panel.pdf"
All paper figures have been compiled into figures/ directory.
